# end-grad-default-ones-like — faded example 3: end_grad Default in a One-Step Reverse Pass

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `end-grad-default-ones-like`. Running the beacon reports progress on the `Backprop: end-grad ones_like default` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: end-grad ones_like default` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`end-grad-default-ones-like`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "end-grad-default-ones-like"
DD_SUBTOPIC = "Backprop: end-grad ones_like default"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The resolved end_grad seed is immediately used in the first backward function call of the reverse pass. For `y = x ** 2`, the backward function is `dL/dx = grad_out * 2 * x`. When `grad_out` comes from `ones_like`, the leaf gradient is simply `2 * x`. When `grad_out` comes from an explicit weight vector, the leaf gradient is scaled element-wise.

## Faded exercise 3

Implement `square_backprop(x, end_grad_array)` that manually runs the backward pass for `y = x ** 2`. Given `x` (a 1-D tensor) and `end_grad_array` (a raw tensor of the same shape, already resolved — could be ones_like or explicit), return `dL/dx = end_grad_array * 2 * x`.

**Fill in:** Compute and return dL/dx as end_grad_array multiplied element-wise by 2 and then by x.

In [ ]:
import torch as t

def square_backprop(x: t.Tensor, end_grad_array: t.Tensor) -> t.Tensor:
    raise NotImplementedError()  # TODO: Compute and return dL/dx as end_grad_array multiplied element-wise by 2 and then by x.


def _test():
    import torch as t
    x = t.tensor([1.0, 2.0, 3.0])
    # default: ones_like seed
    ones_seed = t.ones_like(x)
    grad = square_backprop(x, ones_seed)
    # should match torch autograd on x.pow(2).sum().backward()
    x2 = x.clone().requires_grad_(True)
    x2.pow(2).sum().backward()
    assert t.allclose(grad, x2.grad)
    # explicit weighted seed
    w = t.tensor([3.0, 0.5, 2.0])
    grad_w = square_backprop(x, w)
    x3 = x.clone().requires_grad_(True)
    (w * x3.pow(2)).sum().backward()
    assert t.allclose(grad_w, x3.grad)


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def square_backprop(x: t.Tensor, end_grad_array: t.Tensor) -> t.Tensor:
    return end_grad_array * 2 * x
```
</details>